# WarpTorch: Schwarzschild Black Hole Metric Verification
This notebook acts as a mathematical validation pipeline. Since Schwarzschild geometry is a vacuum solution to Einstein's equations, the Stress-Energy tensor should yield exactly zero outside the horizon ($r > r_s$).

In [ ]:
import sys
import os

# Add parent directory to path to import core modules
sys.path.insert(0, os.path.abspath('..'))

In [ ]:
import torch
import plotly.graph_objects as go
import numpy as np

from core.metrics.schwarzschild import get_schwarzschild_metric
from core.solver.energy import get_energy_tensor
from core.visualizer.slicing import get_2d_slice
from core.utils import get_best_device

device = get_best_device()
print(f"Using device: {device}")

## 1. Initialize Black Hole Metric
Setting up a Schwarzschild horizon with radius $r_s = 4.0$ meters at the center of the numerical grid.

In [ ]:
grid_size = (1, 50, 50, 50)
grid_scale = (0.1, 0.4, 0.4, 0.4)
world_center = (0.0, 10.0, 10.0, 10.0)
rs = 4.0

print("Generating Schwarzschild metric...")
bh_metric = get_schwarzschild_metric(
    grid_size=grid_size,
    world_center=world_center,
    rs=rs,
    grid_scale=grid_scale,
    device=device
)

## 2. Execute Solver & Analyze Truncation Errors
We evaluate $T^{\mu\nu}$ to inspect numerical behavior near the horizon coordinate singularity.

In [ ]:
print("Calculating Stress-Energy Tensor around the Event Horizon...")
bh_energy = get_energy_tensor(bh_metric)

## 3. Map Vacuum Precision
Plotting the slice to see where numerical artifacts drop to $0$ far from the black hole.

In [ ]:
t00_bh_slice = get_2d_slice(bh_energy, component=(0, 0), slice_plane='xy')

x_coords = (np.arange(grid_size[1]) * grid_scale[1]) - world_center[1]
y_coords = (np.arange(grid_size[2]) * grid_scale[2]) - world_center[2]

fig_bh = go.Figure(data=go.Heatmap(
    z=t00_bh_slice.T, x=x_coords, y=y_coords,
    colorscale='Hot',
    colorbar=dict(title="Numerical Artifact Density")
))
fig_bh.update_layout(
    title="Schwarzschild Vacuum Verification ($T^{00}$ Field)",
    xaxis_title="X (Meters)", yaxis_title="Y (Meters)"
)
fig_bh.show()